# 01. Xác định bài toán, hướng phân tích và yêu cầu dữ liệu

**Đề tài:** Dự đoán Nutri-Score (A–E) cho sản phẩm thiếu bảng dinh dưỡng, chỉ từ thành phần, phụ gia, danh mục và nhóm NOVA, dữ liệu Open Food Facts.

Bộ notebook đi theo đúng quy trình 9 bước của môn học:

| Bước trong quy trình | Notebook |
|---|---|
| 1. Business Understanding, 2. Analytic Approach, 3. Data Requirements | **01_bai_toan** (notebook này) |
| 4. Data Collection (SQL) | 02_thu_thap_du_lieu |
| 5. Data Understanding / EDA | 03_eda |
| 6. Data Preparation (làm sạch, đặc trưng, chia train/validation/test) | 04_chuan_bi_du_lieu |
| 7. Modeling, 8. Evaluation trên tập dev | 05_mo_hinh |
| 8. Evaluation cuối trên tập test, 9. Deployment | làm sau (tập test đang khóa) |

Mã dùng lại nhiều lần nằm trong `src/` (mỗi tệp ngắn, có chú thích); còn lại viết thẳng trong notebook.

In [1]:
# Chạy từ thư mục gốc dự án để đọc được data/, sql/ và import src/
import os, sys
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(root)
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

## Bước 1. Xác định bài toán (Business Understanding)

**Vấn đề thực tế.** Nutri-Score (A tốt nhất, E kém nhất) giúp người mua so sánh nhanh chất lượng dinh dưỡng, nhưng chỉ tính được khi có **bảng dinh dưỡng**. Trên Open Food Facts, rất nhiều sản phẩm có danh sách thành phần (chụp từ bao bì) nhưng không có bảng dinh dưỡng, nên không có Nutri-Score. Ô dưới đếm trực tiếp trên bản dump đầy đủ.

In [2]:
# Đếm trên toàn bộ bản dump (4,8 triệu sản phẩm) bằng SQL của DuckDB, đọc thẳng tệp Parquet
import duckdb

duckdb.sql('''
    SELECT count(*) AS tong_san_pham,
           count(*) FILTER (WHERE lower(nutriscore_grade) IN ('a','b','c','d','e')) AS co_nutri_score,
           count(*) FILTER (WHERE len(ingredients_tags) > 0
                            AND coalesce(lower(nutriscore_grade), '') NOT IN ('a','b','c','d','e')) AS co_thanh_phan_nhung_khong_co_nutri_score
    FROM read_parquet('data/external/food.parquet')
''').df()

,tong_san_pham,co_nutri_score,co_thanh_phan_nhung_khong_co_nutri_score
0,4815797,1421625,381018


Chỉ khoảng 30% sản phẩm có Nutri-Score; khoảng **381 nghìn sản phẩm có thành phần mà không có Nutri-Score**. Đó là nhóm mô hình hướng tới.

| Mục | Nội dung |
|---|---|
| **Goal** | Ước lượng hạng Nutri-Score cho sản phẩm chưa có bảng dinh dưỡng, để người mua và cộng đồng Open Food Facts có thông tin tham khảo sớm |
| **Objectives** | (1) Xây mô hình phân loại A–E chỉ từ thành phần, phụ gia, danh mục, NOVA, quốc gia; (2) vượt rõ đường cơ sở "đoán theo nhóm danh mục"; (3) chỉ ra thành phần/phụ gia và nhóm đặc trưng ảnh hưởng nhiều nhất |
| **Stakeholders** | Người tiêu dùng (ứng dụng quét mã vạch); cộng đồng Open Food Facts (ưu tiên sản phẩm cần bổ sung bảng dinh dưỡng); nhà bán lẻ, nhà sản xuất nhỏ; nhà nghiên cứu dinh dưỡng |

**Câu hỏi phân tích (đo lường được):**
1. Chỉ từ các đặc trưng trên, dự đoán đúng đến mức nào (Macro-F1, QWK), và nhóm sản phẩm/hạng nào khó nhất?
2. Thành phần và phụ gia nào kéo hạng xuống D/E mạnh nhất?
3. Nhóm đặc trưng nào đóng góp nhiều nhất, và NOVA bổ sung bao nhiêu so với riêng thành phần?

**Tiêu chí thành công:**

| Tiêu chí | Ngưỡng | Vì sao |
|---|---|---|
| Vượt đường cơ sở "hạng phổ biến nhất của nhóm danh mục `pnns_groups_2`" | Macro-F1 > 0,412 **và** QWK > 0,463 (5 fold trên dev) | Mô hình phải hơn một quy tắc đơn giản ai cũng nghĩ ra |
| Sai không quá xa | ≥ 85% dự đoán lệch tối đa 1 hạng | Đoán B thành C ít hại hơn đoán A thành E |
| Kết quả khách quan | Đo **một lần** trên tập test 20% đã khóa; chênh với dev nhỏ | Tránh tự lừa mình khi chọn mô hình trên chính dữ liệu đo |
| Không gian lận | Không dùng bảng dinh dưỡng hay thẻ lộ đáp án làm đặc trưng | Sản phẩm thật sự thiếu dinh dưỡng sẽ không có các thông tin này |

## Bước 2. Hướng phân tích (Analytic Approach)

| Câu hỏi | Dạng bài toán | Cách làm |
|---|---|---|
| Dữ liệu trông thế nào, hạng phân bố ra sao? | Descriptive | Thống kê mô tả, biểu đồ (notebook 03) |
| Vì sao sản phẩm bị D/E? | Diagnostic | Lift của thành phần/phụ gia, NOVA × hạng, độ quan trọng đặc trưng |
| Sản phẩm mới thuộc hạng nào? | **Predictive – Classification 5 lớp có thứ tự** (A < B < C < D < E) | Mô hình phân loại có giám sát |

**Vì sao là phân loại, không phải hồi quy?** Đầu ra cần dùng là chữ A–E; điểm Nutri-Score gốc tính từ bảng dinh dưỡng (thứ ta không có). Thứ tự giữa các hạng vẫn được tính đến qua chỉ số QWK và tỷ lệ lệch ≤ 1 hạng.

**Thuật toán dự kiến:** 2 đường cơ sở (đoán hạng phổ biến nhất; đoán theo nhóm danh mục) → Logistic Regression → Decision Tree → Random Forest → LightGBM (gradient boosting), tinh chỉnh bằng Optuna.

**Chỉ số:** Macro-F1 là chỉ số chính (5 hạng quan trọng như nhau dù E nhiều gấp 2,6 lần B); thêm Accuracy, Precision, Recall, QWK, ROC-AUC, tỷ lệ lệch ≤ 1 hạng.

**Cách kiểm định:** chia theo **thương hiệu** (StratifiedGroupKFold), vì sản phẩm cùng hãng rất giống nhau; chia ngẫu nhiên từng dòng sẽ cho điểm cao ảo.

## Bước 3. Yêu cầu dữ liệu (Data Requirements)

| Nhóm biến | Trường (Open Food Facts) | Kiểu dữ liệu | Vai trò |
|---|---|---|---|
| Mã định danh | `code` | chuỗi (giữ số 0 đầu) | khóa, không làm đặc trưng |
| Thành phần | `ingredients_tags`, `ingredients_n`, `ingredients_from_palm_oil_n`, `ingredients_analysis_tags`, `ingredients_text` | danh sách thẻ, số, chữ | **đặc trưng** |
| Phụ gia | `additives_tags`, `additives_n` | danh sách thẻ, số | **đặc trưng** |
| Danh mục | `categories_tags`, `pnns_groups_1`, `pnns_groups_2` | danh sách thẻ, phân loại | **đặc trưng** |
| Chế biến | `nova_group` | phân loại 1–4 | **đặc trưng** |
| Nhãn bao bì, quốc gia | `labels_tags`, `countries_tags` | danh sách thẻ | **đặc trưng** (bỏ thẻ lộ đáp án) |
| Thương hiệu, tên | `brands_tags`, `product_name` | danh sách thẻ, chữ | chỉ để tạo nhóm khi chia dữ liệu và tìm trùng |
| **Nhãn (y)** | `nutriscore_grade` | phân loại a–e | biến cần dự đoán |
| Dinh dưỡng, điểm | `nutriments`, `nutriscore_score` | số | **CHỈ cho EDA**, cấm làm đặc trưng (lộ đáp án) |
| Trường cá nhân | `creator`, `editors_tags`, ... | — | **không thu thập** |

**Nguồn:** bản dump Parquet chính thức `openfoodfacts/product-database` trên Hugging Face (giấy phép ODbL), lấy mẫu bằng SQL rồi lưu vào PostgreSQL. (Search API đã thử và bỏ: giới hạn 10 request/phút, hay lỗi 503, mẫu không ngẫu nhiên.)

**Khối lượng và chất lượng cần:** ≥ 10.000 sản phẩm có nhãn A–E và có thành phần; mỗi hạng ≥ 100 mẫu; NOVA phủ ≥ 50%; thẻ thành phần đã chuẩn hóa (tiền tố `en:`) ≥ 90%; không thương hiệu nào chiếm > 10%. Chọn **30.000 sản phẩm ngẫu nhiên (seed 42)** để giữ phân bố hạng tự nhiên. Notebook 03 kiểm tra từng yêu cầu này bằng SQL.

Danh sách cột thật sự được đọc từ bản dump và danh sách trường cá nhân bị chặn nằm trong mã:

In [3]:
from src.fetch_from_dump import COLUMNS
from src.cleaning import PII_FIELDS

print("Cột đọc từ bản dump:", COLUMNS)
print("Trường cá nhân bị chặn:", sorted(PII_FIELDS))
print("Có trường cá nhân nào trong danh sách đọc không?", bool(set(COLUMNS) & PII_FIELDS))

Cột đọc từ bản dump: ['code', 'product_name', 'brands_tags', 'countries_tags', 'categories_tags', 'food_groups_tags', 'ingredients_tags', 'ingredients_text', 'ingredients_n', 'ingredients_from_palm_oil_n', 'ingredients_analysis_tags', 'additives_tags', 'additives_n', 'allergens_tags', 'labels_tags', 'nova_group', 'nutriscore_grade', 'nutriscore_score', 'nutriments', 'lang', 'last_modified_t']
Trường cá nhân bị chặn: ['checkers_tags', 'correctors_tags', 'creator', 'editors_tags', 'informers_tags', 'last_editor', 'photographers_tags']
Có trường cá nhân nào trong danh sách đọc không? False
